In [ ]:
import sys
from pathlib import Path
import os

sys.path.append("..")

In [ ]:
import importlib
import dataset.load_dataset as load_dataset

In [ ]:
importlib.reload(load_dataset)
from dataset.load_dataset import SROIEDataset

import torch

# load image imports
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
from src.utils.image_utils import read_box_file, draw_boxes, visualize_token_grid

# import the VLM model and processor
from qwen_vl_utils import process_vision_info
from src.vlm_model import load_vlm_model, load_processor
from dataset.custom_multimodal_dataset import SROIEMultimodalDataset

#### uncomment below cell if passing the dataset dir

In [ ]:
# # Detect if running in Google Colab
# IN_COLAB = "google.colab" in str(get_ipython())
# if IN_COLAB:
#     from google.colab import drive
#     drive.mount('/content/drive')
#     DATASET_DIR = Path("/content/drive/MyDrive/datasets/my_dataset")
# else:
#     # "/Users/yourname/datasets/my_dataset" should be replaced with the actual path to your dataset on your local machine
#     DATASET_DIR = Path("/Users/yourname/datasets/my_dataset")
# print("Dataset path:", DATASET_DIR)
# data_splits_path = DATASET_DIR / "versions/1"

In [ ]:
sroie_dataset = SROIEDataset(resize_offline=True)
sroie_dataset.train.head()

### Load the model and the processor

In [ ]:
model = load_vlm_model()
processor = load_processor()

In [ ]:
sroi_custom_dataset = SROIEMultimodalDataset(data_frame=sroie_dataset.train, split='train', processor=processor)

In [ ]:
sample_row = sroi_custom_dataset.data_frame.iloc[100]

In [ ]:
image_sroie = Image.open(sample_row["img_path"])
plt.figure(figsize=(12, 12))
plt.imshow(image_sroie)
plt.axis("off");

In [ ]:
sroi_custom_dataset[100]

In [ ]:
visualize_token_grid(image_path=sample_row["img_path"], token_count=728)

### Testing with Zero-shot prompt

In [ ]:
image = Image.open(sample_row["img_path"]).convert("RGB")
print(f"Original image size: {image.size}")

# lets us define the Zero-Shot Prompt
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Extract the Company, Date, Address, and Total from this receipt. Output ONLY a valid JSON object."},
        ],
    }
]

# infernce required processing steps
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)

# lets move all input tensors to the model's device GPU
inputs = {k: v.to(model.device) for k, v in inputs.items()}
# NOTE: this line should remain commented out if you are running on GPU, but if you are on Mac with MPS, you should uncomment it and comment out the line above
#inputs = inputs.to("mps")

print("Generating baseline response...")
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=256)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs['input_ids'], generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

print("\n--- Zero-Shot Output ---")
print(output_text)